# Task 4 — Dataset Exploration: Oxford-102 Flowers

**Goal:** Load and examine the Oxford-102 Flowers dataset. Analyze dataset statistics
(number of classes, images per class, image resolution) and explore/display text
descriptions combined with photos.

**Note on captions:** Oxford-102 does not ship with natural-language captions by
default (unlike COCO). We generate simple templated captions from the 102 class
names (e.g. *"a photo of a pink primrose"*) so later tasks (3, 5, 6) have text
descriptions paired with images. This is a well-known and standard workaround for
this dataset and is documented here transparently.

**Environment:** Google Colab (free tier, T4 GPU not required for this notebook —
CPU is enough for exploration).

## 1. Setup

In [ ]:
!pip install -q torchvision matplotlib pandas seaborn

import torch
import torchvision
from torchvision.datasets import Flowers102
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from PIL import Image
import random

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)

## 2. (Optional) Mount Google Drive

Run this so downloaded data / any outputs persist across Colab sessions.
You can skip this cell if you don't want to use Drive yet — the dataset will
just re-download next session (it's small, ~330MB).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_ROOT = '/content/drive/MyDrive/elevance-skills/data'
os.makedirs(DATA_ROOT, exist_ok=True)
print('Data will be stored at:', DATA_ROOT)

If you'd rather not use Drive right now, just run this cell instead of the one above:
```python
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)
```

## 3. Load the Oxford-102 Flowers dataset

`torchvision.datasets.Flowers102` downloads directly from the official Oxford VGG
source and handles the train/val/test split and image-to-label mapping for us.

In [ ]:
train_set = Flowers102(root=DATA_ROOT, split='train', download=True)
val_set   = Flowers102(root=DATA_ROOT, split='val', download=True)
test_set  = Flowers102(root=DATA_ROOT, split='test', download=True)

print(f'Train images: {len(train_set)}')
print(f'Val images:   {len(val_set)}')
print(f'Test images:  {len(test_set)}')
print(f'Total images: {len(train_set) + len(val_set) + len(test_set)}')

## 4. Class names

`torchvision` gives us numeric labels (0–101) but not the human-readable flower
names. We hardcode the well-known official Oxford-102 class name list here, ordered
to match the dataset's label indices.

In [ ]:
# Official Oxford-102 class names, index-aligned with torchvision's labels (0-101)
CLASS_NAMES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea',
    'english marigold', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood',
    'globe thistle', 'snapdragon', "colt's foot", 'king protea', 'spear thistle',
    'yellow iris', 'globe flower', 'purple coneflower', 'peruvian lily', 'balloon flower',
    'giant white arum lily', 'fire lily', 'pincushion flower', 'fritillary', 'red ginger',
    'grape hyacinth', 'corn poppy', 'prince of wales feathers', 'stemless gentian',
    'artichoke', 'sweet william', 'carnation', 'garden phlox', 'love in the mist',
    'mexican aster', 'alpine sea holly', 'ruby-lipped cattleya', 'cape flower',
    'great masterwort', 'siam tulip', 'lenten rose', 'barbeton daisy', 'daffodil',
    'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower', 'marigold',
    'buttercup', 'oxeye daisy', 'common dandelion', 'petunia', 'wild pansy', 'primula',
    'sunflower', 'pelargonium', 'bishop of llandaff', 'gaura', 'geranium',
    'orange dahlia', 'pink-yellow dahlia', 'cautleya spicata', 'japanese anemone',
    'black-eyed susan', 'silverbush', 'californian poppy', 'osteospermum',
    'spring crocus', 'bearded iris', 'windflower', 'tree poppy', 'gazania',
    'azalea', 'water lily', 'rose', 'thorn apple', 'morning glory', 'passion flower',
    'lotus', 'toad lily', 'anthurium', 'frangipani', 'clematis', 'hibiscus',
    'columbine', 'desert-rose', 'tree mallow', 'magnolia', 'cyclamen', 'watercress',
    'canna lily', 'hippeastrum', 'bee balm', 'ball moss', 'foxglove', 'bougainvillea',
    'camellia', 'mallow', 'mexican petunia', 'bromelia', 'blanket flower',
    'trumpet creeper', 'blackberry lily'
]

print(f'Number of classes: {len(CLASS_NAMES)}')
assert len(CLASS_NAMES) == 102, 'Expected exactly 102 class names'

## 5. Generate templated text captions

We build a small helper that turns a numeric label into a natural-language caption
using a few varying templates, so text isn't perfectly repetitive across images of
the same class (useful later when we tokenize/encode text in Task 3).

In [ ]:
CAPTION_TEMPLATES = [
    'a photo of a {}',
    'a close-up photo of a {} flower',
    'a beautiful {} in bloom',
    'an image showing a {} flower',
    'a {} flower with vivid petals',
]

def make_caption(label_idx: int, seed: int = None) -> str:
    name = CLASS_NAMES[label_idx]
    rng = random.Random(seed)
    template = rng.choice(CAPTION_TEMPLATES)
    return template.format(name)

# quick sanity check
for i in range(5):
    img, label = train_set[i]
    print(f'label={label:3d} -> {make_caption(label, seed=i)}')

## 6. Dataset statistics

### 6.1 Number of images per class

In [ ]:
all_labels = [label for _, label in train_set] + [label for _, label in val_set] + [label for _, label in test_set]
label_counts = Counter(all_labels)

counts_df = pd.DataFrame({
    'class_idx': list(label_counts.keys()),
    'class_name': [CLASS_NAMES[i] for i in label_counts.keys()],
    'count': list(label_counts.values())
}).sort_values('count', ascending=False).reset_index(drop=True)

print('Classes:', len(counts_df))
print('Min images in a class:', counts_df['count'].min())
print('Max images in a class:', counts_df['count'].max())
print('Mean images per class:', round(counts_df['count'].mean(), 2))
counts_df.head(10)

In [ ]:
plt.figure(figsize=(14, 5))
sns.barplot(data=counts_df, x='class_idx', y='count', color='#4C72B0')
plt.title('Number of Images per Class (Oxford-102 Flowers, all splits combined)')
plt.xlabel('Class index')
plt.ylabel('Image count')
plt.xticks([])
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

### 6.2 Image resolution statistics

We sample a subset (checking all ~8000 images would be slow) to estimate the
distribution of image widths and heights, since Oxford-102 images are not
uniformly sized.

In [ ]:
SAMPLE_SIZE = 300
sample_indices = random.sample(range(len(train_set)), min(SAMPLE_SIZE, len(train_set)))

widths, heights = [], []
for idx in sample_indices:
    img, _ = train_set[idx]
    w, h = img.size
    widths.append(w)
    heights.append(h)

res_df = pd.DataFrame({'width': widths, 'height': heights})
print(res_df.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(res_df['width'], bins=30, ax=axes[0], color='#55A868')
axes[0].set_title('Image Width Distribution (sampled)')
sns.histplot(res_df['height'], bins=30, ax=axes[1], color='#C44E52')
axes[1].set_title('Image Height Distribution (sampled)')
plt.tight_layout()
plt.savefig('resolution_distribution.png', dpi=150)
plt.show()

### 6.3 Caption length statistics

Since our captions are templated, this is mostly a sanity check — but it mirrors
the kind of analysis you'd do on a real captioned dataset like COCO.

In [ ]:
sample_captions = [make_caption(label, seed=i) for i, (_, label) in enumerate(train_set)]
caption_lengths = [len(c.split()) for c in sample_captions]

print('Mean caption length (words):', round(np.mean(caption_lengths), 2))
print('Min / Max caption length:', min(caption_lengths), '/', max(caption_lengths))

plt.figure(figsize=(8, 4))
sns.histplot(caption_lengths, bins=range(min(caption_lengths), max(caption_lengths)+2), color='#8172B2')
plt.title('Caption Length Distribution (in words)')
plt.xlabel('Number of words')
plt.tight_layout()
plt.savefig('caption_length_distribution.png', dpi=150)
plt.show()

## 7. Visualize sample images with captions

This is the key deliverable for Task 4: showing images paired with their text
descriptions.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_idxs = random.sample(range(len(train_set)), 8)

for ax, idx in zip(axes.flat, sample_idxs):
    img, label = train_set[idx]
    caption = make_caption(label, seed=idx)
    ax.imshow(img)
    ax.set_title(caption, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_images_with_captions.png', dpi=150)
plt.show()

## 8. Save a lightweight metadata CSV

This will be reused in later tasks (3, 5, 6) so we don't have to redo dataset
loading/captioning logic each time. We save class index, class name, and a
generated caption for every image in each split (paths are re-derived from the
torchvision dataset objects at load time, so we only store label + caption + split + index).

In [ ]:
def build_metadata(dataset, split_name):
    rows = []
    for i, (_, label) in enumerate(dataset):
        rows.append({
            'split': split_name,
            'index_in_split': i,
            'label': label,
            'class_name': CLASS_NAMES[label],
            'caption': make_caption(label, seed=hash((split_name, i)) % (2**31))
        })
    return rows

metadata_rows = (
    build_metadata(train_set, 'train') +
    build_metadata(val_set, 'val') +
    build_metadata(test_set, 'test')
)

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv('oxford102_metadata.csv', index=False)
print(f'Saved metadata for {len(metadata_df)} images.')
metadata_df.head()

## 9. Summary of findings

Fill this in after running the notebook, then copy the key points into
`NOTES.md` and today's daily log. Example points to note:
- Total images across all splits
- Class imbalance (min vs max images per class)
- Typical image resolution range
- Anything surprising you noticed while exploring